# CSI 300 Daily - Exploratory Data Analysis

This notebook explores the CSI 300 Daily stock index dataset.

**Dataset:** `csi300-daily.csv`

**Description:** Daily CSI 300 stock index data including price, open, high, low, volume, and change percentage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Paths
DATA_PATH = Path('../../datasets/raw/csi300-daily.csv')

## 1. Load Data

In [ ]:
# Load the dataset
df = pd.read_csv(DATA_PATH)

# Display first few rows
print("First 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

## 2. Data Cleaning and Preprocessing

In [ ]:
# Clean column names
df.columns = df.columns.str.strip()

# Parse date column
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

# Clean numeric columns (remove commas and convert to float)
numeric_cols = ['Price', 'Open', 'High', 'Low', 'Vol.']
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Clean percentage column
if 'Change %' in df.columns:
    df['Change %'] = df['Change %'].str.replace('%', '').astype(float)

# Sort by date
df = df.sort_values('Date').reset_index(drop=True)

print("Cleaned dataset:")
display(df.head())
print("\nData types:")
print(df.dtypes)

## 3. Data Overview

In [ ]:
# Basic information
print("Dataset Shape:", df.shape)
print("\nDate range:", df['Date'].min(), "to", df['Date'].max())
print("Total trading days:", len(df))
print("\nDataset Info:")
df.info()

## 4. Missing Values Analysis

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})

print("Missing Values Summary:")
display(missing_df[missing_df['Missing Count'] > 0])

if missing.sum() == 0:
    print("\n✓ No missing values found!")

## 5. Descriptive Statistics

In [ ]:
# Statistical summary
print("Descriptive Statistics:")
display(df[['Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']].describe())

## 6. Price Time Series Analysis

In [ ]:
# Plot closing price over time
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['Date'], df['Price'], linewidth=1, alpha=0.8)
ax.set_title('CSI 300 Index - Closing Price Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot with moving averages
df['MA_50'] = df['Price'].rolling(window=50).mean()
df['MA_200'] = df['Price'].rolling(window=200).mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['Date'], df['Price'], linewidth=1, alpha=0.6, label='Price')
ax.plot(df['Date'], df['MA_50'], linewidth=2, label='50-day MA', color='orange')
ax.plot(df['Date'], df['MA_200'], linewidth=2, label='200-day MA', color='red')
ax.set_title('CSI 300 Index with Moving Averages', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Candlestick Visualization (OHLC)

In [ ]:
# Plot recent OHLC data
recent_df = df.tail(100).copy()

fig, ax = plt.subplots(figsize=(16, 6))

# Plot high-low range
for idx, row in recent_df.iterrows():
    color = 'green' if row['Price'] >= row['Open'] else 'red'
    ax.plot([row['Date'], row['Date']], [row['Low'], row['High']], color=color, linewidth=1, alpha=0.7)
    ax.plot([row['Date'], row['Date']], [row['Open'], row['Price']], color=color, linewidth=3, alpha=0.9)

ax.set_title('CSI 300 - Recent 100 Days OHLC', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Daily Returns Analysis

In [ ]:
# Calculate daily returns
df['Daily_Return'] = df['Price'].pct_change() * 100

# Plot daily returns
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Returns over time
axes[0].plot(df['Date'], df['Daily_Return'], linewidth=0.5, alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Daily Returns Over Time', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Daily Return (%)')
axes[0].grid(True, alpha=0.3)

# Distribution of returns
axes[1].hist(df['Daily_Return'].dropna(), bins=100, edgecolor='black', alpha=0.7)
axes[1].axvline(df['Daily_Return'].mean(), color='red', linestyle='--', label=f'Mean: {df["Daily_Return"].mean():.4f}%')
axes[1].axvline(df['Daily_Return'].median(), color='green', linestyle='--', label=f'Median: {df["Daily_Return"].median():.4f}%')
axes[1].set_title('Distribution of Daily Returns', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Daily Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Daily Returns Statistics:")
print(f"Mean: {df['Daily_Return'].mean():.4f}%")
print(f"Std Dev: {df['Daily_Return'].std():.4f}%")
print(f"Min: {df['Daily_Return'].min():.4f}%")
print(f"Max: {df['Daily_Return'].max():.4f}%")

## 9. Volatility Analysis

In [ ]:
# Calculate rolling volatility (30-day)
df['Volatility_30d'] = df['Daily_Return'].rolling(window=30).std()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['Date'], df['Volatility_30d'], linewidth=1, color='purple')
ax.set_title('30-Day Rolling Volatility', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Volatility (Std Dev of Returns)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Volume Analysis

In [ ]:
# Plot trading volume
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Price
axes[0].plot(df['Date'], df['Price'], linewidth=1)
axes[0].set_title('CSI 300 Price', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price')
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(df['Date'], df['Vol.'], width=1, alpha=0.7)
axes[1].set_title('Trading Volume', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Volume (K)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Correlation Analysis

In [ ]:
# Correlation matrix
corr_cols = ['Price', 'Open', 'High', 'Low', 'Vol.', 'Change %', 'Daily_Return']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Summary and Key Findings

In [ ]:
print("=" * 60)
print("KEY FINDINGS - CSI 300 DAILY")
print("=" * 60)
print(f"\n1. Dataset Coverage:")
print(f"   - Start Date: {df['Date'].min().strftime('%Y-%m-%d')}")
print(f"   - End Date: {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"   - Total Trading Days: {len(df):,}")

print(f"\n2. Price Statistics:")
print(f"   - Current Price: {df['Price'].iloc[-1]:.2f}")
print(f"   - All-time High: {df['Price'].max():.2f} ({df.loc[df['Price'].idxmax(), 'Date'].strftime('%Y-%m-%d')})")
print(f"   - All-time Low: {df['Price'].min():.2f} ({df.loc[df['Price'].idxmin(), 'Date'].strftime('%Y-%m-%d')})")
print(f"   - Average Price: {df['Price'].mean():.2f}")

print(f"\n3. Returns Analysis:")
print(f"   - Average Daily Return: {df['Daily_Return'].mean():.4f}%")
print(f"   - Daily Volatility: {df['Daily_Return'].std():.4f}%")
print(f"   - Best Day: {df['Daily_Return'].max():.2f}% ({df.loc[df['Daily_Return'].idxmax(), 'Date'].strftime('%Y-%m-%d')})")
print(f"   - Worst Day: {df['Daily_Return'].min():.2f}% ({df.loc[df['Daily_Return'].idxmin(), 'Date'].strftime('%Y-%m-%d')})")

print(f"\n4. Volume Statistics:")
print(f"   - Average Volume: {df['Vol.'].mean():.2f}K")
print(f"   - Max Volume: {df['Vol.'].max():.2f}K ({df.loc[df['Vol.'].idxmax(), 'Date'].strftime('%Y-%m-%d')})")

print(f"\n5. Data Quality:")
print(f"   - Missing Values: {df.isnull().sum().sum()}")

print("\n" + "=" * 60)